In [1]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, DensityMatrix, partial_trace, concurrence, purity
from qiskit_aer import AerSimulator
# Conceptual QKD (E91-style): Alice and Bob each pick a random basis (Z or X)
# to measure their half of a shared Phi+ pair; only matching-basis results become key bits.
rng = np.random.default_rng(42)
sim = AerSimulator()
N = 500
alice_bases = rng.integers(0, 2, N)  # 0 = Z basis, 1 = X basis
bob_bases = rng.integers(0, 2, N)
alice_bits, bob_bits = [], []

for i in range(N):
    qc = QuantumCircuit(2, 2)
    qc.h(0); qc.cx(0, 1)  # shared Phi+ pair
    if alice_bases[i] == 1:
        qc.h(0)
    if bob_bases[i] == 1:
        qc.h(1)
    qc.measure([0, 1], [0, 1])
    outcome = list(sim.run(qc, shots=1).result().get_counts().keys())[0]
    bob_bit, alice_bit = int(outcome[0]), int(outcome[1])
    alice_bits.append(alice_bit); bob_bits.append(bob_bit)

alice_bits, bob_bits = np.array(alice_bits), np.array(bob_bits)
same_basis = alice_bases == bob_bases

print("Bell pairs distributed:", N)
print("Pairs where Alice and Bob happened to pick the same basis:", same_basis.sum())
print("Agreement when bases MATCH (usable key bits): "
      f"{np.mean(alice_bits[same_basis] == bob_bits[same_basis]):.1%}")
print("Agreement when bases DIFFER (discarded):        "
      f"{np.mean(alice_bits[~same_basis] == bob_bits[~same_basis]):.1%}")
print("Sifted key length:", same_basis.sum(), f"({same_basis.sum()/N:.1%} of pairs)")
print("Alice's sifted key (first 32 bits):", ''.join(map(str, alice_bits[same_basis][:32])))
print("Bob's sifted key   (first 32 bits):", ''.join(map(str, bob_bits[same_basis][:32])))


Bell pairs distributed: 500
Pairs where Alice and Bob happened to pick the same basis: 253
Agreement when bases MATCH (usable key bits): 100.0%
Agreement when bases DIFFER (discarded):        55.9%
Sifted key length: 253 (50.6% of pairs)
Alice's sifted key (first 32 bits): 00111111010011101111111000010000
Bob's sifted key   (first 32 bits): 00111111010011101111111000010000
